In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb

util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [ ]:
#Step2 - Read TBI_EPI_Cohort
TBI_EPI_Cohort_PersonId_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Cohort_PersonId_Final')

In [ ]:
epi_med = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/epi_med')

In [ ]:
epi_med.printSchema()

In [ ]:
epi_med.createOrReplaceTempView('EPI_Medication_Table')

In [ ]:
#Step3 - Read processed Medication parquet
Med_Cohort = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/medication_s.parquet')

In [ ]:
TBI_EPI_Cohort_PersonId_Final.createOrReplaceTempView('TBI_EPI_Cohort')

In [ ]:
TBI_EPI_Med_result_1 = spark.sql("""
    SELECT COUNT(DISTINCT t.personid) AS distinct_personid_count
    FROM TBI_EPI_Cohort t
    LEFT JOIN EPI_Medication_Table e ON t.personid = e.personid
    WHERE e.personid IS NULL

""")
TBI_EPI_Med_result_1.show()

In [ ]:
TBI_EPI_Extract = spark.sql("""
    SELECT COUNT(DISTINCT(personid)) from TBI_EPI_Cohort
""")
TBI_EPI_Extract.show()

In [ ]:
TBI_NOT_EPI = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NOT_EPI_Cohort_PersonId_Final')

In [ ]:
TBI_NOT_EPI.createOrReplaceTempView('TBI_NOT_EPI')

In [ ]:
Total_med_Cohort = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Med_Final.parquet')

In [ ]:
Total_med_Cohort.createOrReplaceTempView('EPI_MED_Cohort')

In [ ]:
EPI_MED_Extract = spark.sql("""
    SELECT COUNT(DISTINCT(personid)) from EPI_MED_Cohort
""")
EPI_MED_Extract.show()

In [ ]:
TBI_EPI_MED_Check = spark.sql("""SELECT COUNT(DISTINCT t.personid) AS distinct_personid_count FROM TBI_EPI_Cohort t LEFT JOIN EPI_MED_Cohort e ON t.personid = e.personid
WHERE e.personid IS NULL""")
TBI_EPI_MED_Check.show()

In [ ]:
TBI_with_EPI_Total_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_cohort')

In [ ]:
TBI_with_EPI_Total_Final.createOrReplaceTempView('TBI_EPI')

In [ ]:
TBI = spark.sql("""
select * from TBI_EPI
where conditioncode in ('Z87.820','S02.1','R56.1') or
conditioncode like 'S06%' or
conditioncode like 'S07%' or
conditioncode like 'S08%' or
conditioncode like 'S09%' or
conditioncode like 'G44.3%' or
conditioncode like '854.%' or
conditioncode like '851.%' or
conditioncode like '852.%' or
conditioncode like '853.%'
""")

In [ ]:
TBI.createOrReplaceTempView('TBI_Total')

In [ ]:
TBI_Total = spark.sql("""SELECT COUNT(DISTINCT(personid)) from TBI_Total""")
TBI_Total.show()

In [ ]:
# TBI_EPI_Med_result = spark.sql("""
#     SELECT e.personid, e.startdate AS date
#     FROM EPI_Medication_Table e
#     INNER JOIN TBI_EPI t ON e.personid = t.personid
# """)
# TBI_EPI_Med_result.show(5, truncate = False)
# Time Consuming -> dont use
TBI_EPI_MED_Final = spark.sql("""SELECT COUNT(DISTINCT t.personid) AS distinct_personid_count FROM TBI_EPI t INNER JOIN EPI_Medication_Table e where t.personid <> e.personid""")
TBI_EPI_MED_Final.show()


In [ ]:
TBI_NOT_EPI_MED_Not_Final = spark.sql("""SELECT COUNT(DISTINCT t.personid) AS distinct_personid_count FROM TBI_NOT_EPI t LEFT JOIN EPI_Medication_Table e ON t.personid = e.personid
WHERE e.personid IS NULL""")
TBI_NOT_EPI_MED_Not_Final.show()

In [ ]:
TBI_NOT_EPI_MED_Final = spark.sql("""SELECT DISTINCT t.personid FROM TBI_NOT_EPI t LEFT JOIN EPI_Medication_Table e ON t.personid = e.personid
WHERE e.personid IS NOT NULL""")
TBI_NOT_EPI_MED_Final.show(5)

In [ ]:
#write - TBI Not Having EPI present in EPI_Medication list to parquet
TBI_NOT_EPI_MED_Final.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NOT_EPI_In_Med_Cohort_Final")

In [ ]:
TBI_EPI_Med_result = spark.sql("""
    SELECT e.personid, e.startdate AS date
    FROM EPI_Medication_Table e
    INNER JOIN TBI_EPI t ON e.personid = t.personid
""")

In [ ]:
TBI_EPI_MED_Final = spark.sql("""SELECT COUNT(DISTINCT t.personid) AS distinct_personid_count FROM TBI_Total t INNER JOIN EPI_MED_Cohort e where t.personid <> e.personid""")
TBI_EPI_MED_Final.show()

In [ ]:
TBI_EPI_MED_Final = spark.sql("""SELECT t.personid, t.date, e.startdate FROM TBI_Total t LEFT JOIN medication e where t.personid = e.personid""")
TBI_EPI_MED_Final.show()

In [ ]:
table_schema = spark.sql(f"DESCRIBE {'real_world_data_jun_2022'}.{'medication'}")

# Show the schema
table_schema.show(truncate=False)

In [ ]:
selected_med = spark.sql("""
select * from medication where drugcode.standard.primaryDisplay like '%arbamazepine%'
""")

In [ ]:
selected_med.show(1)

In [2]:
#write - TBI Not Having EPI present in EPI_Medication list -> Read
TBI_NOT_EPI_MED_Result = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NOT_EPI_In_Med_Cohort_Final")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
TBI_NOT_EPI_MED_Result.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)



In [ ]:
TBI_NOT_EPI_MED_Result.createOrReplaceTempView('TBI_NOT_EPI_MED')

In [ ]:
TBI_NOT_EPI_MED_FINAL = spark.sql("""
  SELECT t.personid, t.startdate
  FROM EPI_Medication_Table t
  LEFT JOIN TBI_NOT_EPI_MED e ON t.personid = e.personid
  WHERE e.personid IS NOT NULL
""")

TBI_NOT_EPI_MED_FINAL.show()

In [ ]:
TBI_NOT_EPI_MED_FINAL.createOrReplaceTempView('TBI_EPI_MED_Cohort_Updated')

In [ ]:
TBI_EPI_MED_Cohort_Update = spark.sql("""
  SELECT personid, min(startdate) FROM TBI_EPI_MED_Cohort_Updated group by personid
""")
TBI_EPI_MED_Cohort_Update.createOrReplaceTempView('TBI_EPI_MED_Cohort_Updated_1')
TBI_EPI_MED_Cohort_Update.show()
# TBI_EPI_Cohort_UpdatedCount = spark.sql("""
#   SELECT COUNT(personid) as count_of_records FROM TBI_EPI_MED_Cohort_Updated_1
# """)

# TBI_EPI_Cohort_UpdatedCount.show()

In [ ]:
TBI_EPI_MED_Cohort_Update1 = spark.sql("""
  SELECT personid, COALESCE(min(startdate), NULL) as date
  FROM TBI_EPI_MED_Cohort_Updated
  GROUP BY personid
""")

TBI_EPI_MED_Cohort_Update1.show()

In [ ]:
TBI_EPI_MED_Cohort_Update1.createOrReplaceTempView('TBI_EPI_MED_Cohort_Updated_Table')

In [ ]:
TBI_EPI_Cohort_UpdatedCount = spark.sql("""
  SELECT COUNT(personid) as count_of_records FROM TBI_EPI_MED_Cohort_Updated_Table
""")

TBI_EPI_Cohort_UpdatedCount.show()

In [ ]:
TBI_EPI_MED_Cohort_Update1.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NOT_EPI_MED_FINAL_Version")

In [4]:
#TBI Having EPI+TBI Having EPI in Medication Table -> Read
TBI_Having_EPI_MED_Result = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Cohort_Final_Updated_1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
TBI_Having_EPI_MED_Result.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- date: string (nullable = true)



In [6]:
print(TBI_Having_EPI_MED_Result.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

102687


<IPython.core.display.Javascript object>

In [ ]:
#TBI Not Having EPI present in EPI_Medication list -> Read
TBI_NOT_EPI_MED_Result = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NOT_EPI_MED_FINAL_Version")

In [ ]:
# Register the DataFrames as temporary tables
TBI_Having_EPI_MED_Result.createOrReplaceTempView("TBI_Having_EPI_MED_Result")
TBI_NOT_EPI_MED_Result.createOrReplaceTempView("TBI_NOT_EPI_MED_Result")

# Use UNION ALL to combine both tables
merged_result = spark.sql("""
  SELECT * FROM TBI_Having_EPI_MED_Result
  UNION ALL
  SELECT * FROM TBI_NOT_EPI_MED_Result
""")

# Show the merged DataFrame
merged_result.show()

In [ ]:
merged_result.createOrReplaceTempView("Merge_Result")

In [ ]:
merged_result.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_MED_Cohort_Final_Version")

In [19]:
#TBI Having EPI+TBI and having EPI in Medication Table -> Read
Merged_Result = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_MED_Cohort_Final_Version")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
Merged_Result.createOrReplaceTempView("Merge_Result")

▸,:,


In [10]:
# SQL Query to count duplicate personids
# Merge_UpdatedCount = spark.sql("""
#   SELECT COUNT(*) as count_of_duplicate_personids
#   FROM (
#     SELECT personid, COUNT(*) as count
#     FROM Merge_Result
#     GROUP BY personid
#     HAVING count > 1
#   )
# """)

Merge_UpdatedCount = spark.sql("""SELECT personid, COUNT(*) as count FROM Merge_Result GROUP BY personid
HAVING count > 1""")
# Show the result
Merge_UpdatedCount.show()

+--------+-----+
|personid|count|
+--------+-----+
+--------+-----+



In [21]:
##extracted_date_df.createOrReplaceTempView('extracted_date_df')
cohort_demo =spark.sql("""
select distinct l.personid, l.birthdate, l.gender,l.race
from
dedupedemographics l
inner join
Merge_Result r
on l.personid = r.personid""")

▸,:,


In [22]:
##Edited bysai testing
from pyspark.sql.functions import col
from pyspark.sql.types import DateType
# Extract the date using regexp_extract
extracted_date_df = cohort_demo.withColumn("gender", col("gender.value")).withColumn("birthdate", col("birthdate.value").cast(DateType())).withColumn("race", col("race.value"))


▸,:,


In [23]:
extracted_date_df.createOrReplaceTempView('extracted_date_df')
cohort_demo =spark.sql("""
select distinct l.personid, l.birthdate, l.gender,l.race
from
extracted_date_df l
inner join
Merge_Result r
on l.personid = r.personid""")

▸,:,


In [24]:
#write -Cohort_demo - Removed duplicates
cohort_demo.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_Stacked_Updated")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
cohort_demo = etl.extractDemo(spark,Merged_Result,outputfilename="cohort_demo_Stacked", outputfolder = "Priya/epilepsy")

▸,:,


select distinct l.personid, l.birthdate, l.gender.standard.primaryDisplay as gender, l.races[1].standard.primaryDisplay as race from demographics l inner join cohort r on l.personid = r.personid


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
cohort_demo_Unprocessed = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_Stacked_Updated")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
cohort_demo_Unprocessed.createOrReplaceTempView("demo_Unprocessed")

▸,:,


In [27]:
# SQL Query to count duplicate personids
# cohort_demo_dup = spark.sql("""
#   SELECT COUNT(*) as count_of_duplicate_personids
#   FROM (
#     SELECT personid, COUNT(*) as count
#     FROM demo_Unprocessed
#     GROUP BY personid
#     HAVING count > 1
#   )
# """)
cohort_demo_dup = spark.sql("""SELECT personid, COUNT(*) as count FROM demo_Unprocessed GROUP BY personid
HAVING count > 1
""")

# Show the result
cohort_demo_dup.show()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-----+
|personid|count|
+--------+-----+
+--------+-----+



<IPython.core.display.Javascript object>

In [28]:
from pyspark.sql.functions import *
demo_processed = etl.process_demo(spark, Merged_Result, cohort_demo_Unprocessed)

▸,:,


In [29]:
demo_processed.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = false)
 |-- conditioncode: string (nullable = false)
 |-- diagnosis_date: string (nullable = true)
 |-- age_at_diagnosis: double (nullable = true)



In [30]:
# Drop the "ID" column
Epilepsy_Cohort_Demo = demo_processed.drop("conditioncode")

▸,:,


In [31]:
Epilepsy_Cohort_Demo.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = false)
 |-- diagnosis_date: string (nullable = true)
 |-- age_at_diagnosis: double (nullable = true)



In [32]:
demo_processed.createOrReplaceTempView('Epilepsy_Cohort_Domo')

▸,:,


In [34]:
Demo_Result = spark.sql("SELECT * FROM Epilepsy_Cohort_Domo where personid ='06b4a13d-ca4a-4111-89bb-07507ff5d0f8'")
Demo_Result.show(5, truncate = False)
# Demo_Result = spark.sql("""SELECT personid, COUNT(*) as count FROM Epilepsy_Cohort_Domo GROUP BY personid HAVING count > 1""")
# Demo_Result.show(5, truncate = False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+----------+------+----------+-------------+-------------------------+----------------+
|personid                            |birthdate |gender|race      |conditioncode|diagnosis_date           |age_at_diagnosis|
+------------------------------------+----------+------+----------+-------------+-------------------------+----------------+
|06b4a13d-ca4a-4111-89bb-07507ff5d0f8|2005-05-11|Female|Other_race|1            |2016-06-24T07:00:00+00:00|11.11906362     |
+------------------------------------+----------+------+----------+-------------+-------------------------+----------------+



<IPython.core.display.Javascript object>

In [4]:
demo_Unprocessed =spark.sql("""
select distinct l.personid, l.birthdate, l.gender,l.race
from
dedupedemographics l""")

▸,:,


In [5]:
##Edited bysai -> Used by Priya
from pyspark.sql.functions import col
from pyspark.sql.types import DateType
# Extract the date using regexp_extract
extracted_date_df = demo_Unprocessed.withColumn("gender", col("gender.value")).withColumn("birthdate", col("birthdate.value").cast(DateType())).withColumn("race", col("race.value"))

▸,:,


In [9]:
cohort_demo = etl.extractUpdatedDemo(spark,Merged_Result,extracted_date_df,outputfilename="cohort_demo_Stacked", outputfolder = "Priya/epilepsy")

▸,:,


select distinct l.personid, l.birthdate, l.gender.standard.primaryDisplay as gender, l.races[1].standard.primaryDisplay as race from demo l inner join cohort r on l.personid = r.personid


AnalysisException: "Can't extract value from gender#17: need struct type but got string; line 1 pos 41"